# 📘 OpenAI API Quick Start

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nawooooojin/openai-api-tutorial-kr/blob/main/1_Text_Generation/00-OpenAI%20API%20Quick%20Start.ipynb)

> **주의:** Colab 런타임을 재시작하면 환경 변수가 초기화되므로 `OPENAI_API_KEY`를 다시 설정해야 합니다.

## 환경 준비 및 API 키 설정

In [ ]:
%pip install -q -U openai

In [1]:
# Colab에서 실행 중이라면 실습 데이터(data/)가 있는 레포를 받아옵니다.
# (로컬에서 레포를 clone 해서 실행 중이라면 그냥 넘어갑니다)
import os, sys

if "google.colab" in sys.modules and not os.path.exists("data"):
    !git clone -q --depth 1 https://github.com/nawooooojin/openai-api-tutorial-kr.git /content/openai-api-tutorial-kr
    %cd /content/openai-api-tutorial-kr/1_Text_Generation

- Colab 메뉴 `보안 비밀(열쇠 아이콘)`에 `OPENAI_API_KEY`를 저장해 두었다면 자동으로 불러옵니다.
- 아래의 코드에서 오류가 난다면 API 키의 오류일 가능성이 높습니다.

In [2]:
import os

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    try:
        from google.colab import userdata  # type: ignore
        OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
    except ImportError:
        pass  # Colab이 아닌 환경
    except Exception:
        print("Colab 메뉴의 보안 비밀(열쇠 아이콘)에 OPENAI_API_KEY가 저장되어 있는지, 노트북 액세스가 켜져 있는지 확인해주세요.")

if not OPENAI_API_KEY:
    from getpass import getpass
    OPENAI_API_KEY = getpass("OPENAI_API_KEY를 입력하세요: ")

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
print("✅ API 키 설정 완료")

✅ API 키 설정 완료


In [3]:
base_dirs = [
    ("data", ["image", "audio", "video"]),
    ("results", ["image", "audio", "video"]),
]

for root, subfolders in base_dirs:
    for sub in subfolders:
        path = os.path.join(root, sub)
        os.makedirs(path, exist_ok=True)
        print(f"ready: {path}")

ready: data/image
ready: data/audio
ready: data/video
ready: results/image
ready: results/audio
ready: results/video


## OpenAI API 시작하기
- 링크: https://platform.openai.com/docs/quickstart
- OpenAI API를 사용하면 챗GPT처럼 프롬프트에 따라 다양한 텍스트(코드, 수식, JSON, 자연어 등)를 간편하게 생성할 수 있습니다.

### 텍스트 생성
- 아래 코드를 실행하면, 프롬프트에 대한 OpenAI API의 응답 결과를 바로 확인할 수 있습니다.

In [4]:
from openai import OpenAI

client = OpenAI()

response = client.responses.create(
    model="gpt-6-luna",
    input="Write a one-sentence bedtime story about a unicorn."
)

print(response.output_text)

Under the moon’s silver glow, a gentle unicorn tucked the stars into the sky and drifted to sleep beneath a blanket of wildflowers.


### 이미지와 파일 분석
- 분류나 시각 요소 감지 등 다양한 분석 작업 수행이 가능합니다.

In [5]:
from openai import OpenAI
import base64

client = OpenAI()


def encode_image(file_path):
    with open(file_path, "rb") as f:
        base64_image = base64.b64encode(f.read()).decode("utf-8")
    return base64_image

cheers = encode_image("data/image/cheers.jpg")

response = client.responses.create(
    model="gpt-6.1-sol",
    input=[
        {
            "role": "user",
            "content": [
                {
                    "type": "input_text",
                    "text": "이 이미지에서 사람들은 무엇을 하고 있나요?",
                },
                {
                    "type": "input_image",
                    "image_url": f"data:image/jpeg;base64,{cheers}"
                }
            ]
        }
    ]
)

print(response.output_text)

세 사람이 식당에서 치킨과 맥주를 즐기며, 웃으면서 맥주잔을 부딪쳐 건배하고 있습니다. 뒤쪽에는 이들의 모습을 사진으로 찍는 사람들도 보입니다.


### 도구(tool)로 모델 확장
- 모델에 도구를 연결해 외부 데이터·함수에 바로 접근할 수 있습니다.
- 기본 제공(웹/파일 검색) 또는 API·코드·서드파티 전용 커스텀 도구 정의 가능합니다.

**웹 검색 예제**

In [6]:
from openai import OpenAI

client = OpenAI()

response = client.responses.create(
    model="gpt-6.1-sol",
    tools=[{"type": "web_search"}],
    input="오늘 있었던 좋은 뉴스를 알려줘."
)

print(response.output_text)

기분이 조금 밝아질 만한 소식들을 찾아봤어요. **오늘(2026년 10월 26일) 보도되거나 진행 중인 소식 3가지**예요. 보도일과 실제 일어난 날이 다른 경우는 따로 표시했어요.

### 🩺 췌장암을 더 일찍 발견할 가능성이 커졌어요
오늘 소개된 메이요클리닉 연구에서 AI가 기존 CT 영상의 미세한 변화를 분석해, 나중에 췌장암 진단을 받은 사례의 **73%를 진단 약 16개월 전 시점의 영상에서 찾아냈어요.** 아직 추가 임상 검증이 필요하지만, 조기 발견을 위한 희망적인 진전이에요. 연구 자체는 지난 4월 발표됐습니다. ([sciencedaily.com](https://www.sciencedaily.com/releases/2026/10/261024222031.htm))

### 🌿 내전으로 훼손됐던 지역에서 야생동물이 돌아오고 있어요
오늘 보도된 모잠비크 잠베지 삼각주 소식이에요. 밀렵 방지와 야생동물 복원, 지역사회 지원을 함께 추진한 사업이 **2026 마르코르상**을 받았어요. 실제 수상일은 **10월 22일**이며, 현지 생물다양성 재단도 이 지역의 상당한 야생동물 회복을 기록했습니다. 다만 사업 재원에 포함된 트로피 사냥 방식에는 논쟁이 있어요. ([getaway.co.za](https://www.getaway.co.za/news/zambezi-delta-conservation-project-receives-international-award/?utm_source=openai))

### 🎨 그림이 지역 문화공간을 돕는 기부로 이어지고 있어요
미국 플로리다의 캐럴우드 문화센터에서는 오늘도 화가 한나 라호츠카의 작품을 활용한 **온라인 자선 전시·판매**가 진행 중이에요. 작품 판매 수익 전액이 문화센터를 지원하는 데 쓰입니다. 거창한 사건은 아니지만, 예술로 동네의 문화공간을 지키는 따뜻한 소식이에요. ([carrollwoodcenter.org](https://carrollwoodcenter.org/calendar/give-thanks-give

### 스트림 응답
서버 전송 스트리밍 이벤트를 사용해 결과가 생성되는 즉시 표시할 수 있습니다.

In [7]:
from openai import OpenAI

client = OpenAI()

stream = client.responses.create(
    model="gpt-6-luna",
    input=[
        {
            "role": "user",
            "content": "'간장 공장 공장장은 강 공장장'을 열 번 빠르게 말해봐.",
        },
    ],
    stream=True,
)


for event in stream:
    if event.type == "response.output_text.delta":
        print(event.delta, end="")

간장 공장 공장장은 강 공장장, 간장 공장 공장장은 강 공장장, 간장 공장 공장장은 강

 공장장, 간장 공장 공장장은 강 공장장, 간장 공장 공장장은 강 공장장, 간장 공장

 공장장은 강 공장장, 간장 공장 공장장은 강 공장장, 간장 공장 공장장은 강 공장장, 간장 공장 공장장은 강

 공장장, 간장 공장 공장장은 강 공장장!

### 참고 자료
- https://platform.openai.com/docs/quickstart